In [1]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [2]:
# queryname = 'Arabidopsis_thaliana'
# queryname = "Erigeron_breviscapus"
queryname = "Glycine_max"
# queryname = 'Zea_mays'

p450_name	complex_name	p450_score	affinity	cnn_score	cnn_affi	wei_score	dock_num	if_right

In [3]:
data = pd.read_csv(our_data + f"screening2/{queryname}_gnina.txt", sep="\t", header=0)
cache = pd.DataFrame()
cache["enzyme"] = data["complex_name"].str.split("_").str[0:-2].str.join("_")
# cache['enzyme'] = data['complex_name'].str.split('_').str[0]
cache["cc"] = data["complex_name"].str.split("_").str[-1]
# cache['enzymecc'] = cache['enzyme'] + cache['cc']
cache["substrate"] = data["complex_name"].str.split("_").str[-2]
data = pd.concat([data, cache], axis=1)

In [4]:
df = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", f"screening2/{queryname}_scores.pkl")
)

In [5]:
print(df["enzyme"].nunique())
print(df["substrate"].nunique())

342
7


In [6]:
df[df["enzyme"] == "CYP72A69"]

,enzyme,sequence,substrate,ESM1b,ECFP,Binding,scores
14,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,SOY,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000000000000000000000001001000000000...,0,0.116024
15,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,NAR,"[-0.11592403054237366, 0.1274290382862091, -0....",0100000000000000000000000000000000000000000000...,0,0.000277
16,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,fluometuron,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000001000000000000000001000000000000...,0,0.101731
17,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,LIQ,"[-0.11592403054237366, 0.1274290382862091, -0....",0100000000000000000000000000000000000000000000...,0,0.000417
18,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,DIT,"[-0.11592403054237366, 0.1274290382862091, -0....",0001000000000001000000000000000000000000000000...,0,0.001888
19,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,BAM,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000000000000000000000101001000000000...,0,0.860992
20,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,DAI,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000000000000000000000000000000000000...,0,0.001844
2394,CYP72A69,MEAAWVNILMLILILALIWVWKKFNSLWLTPKRLEKILREQGLRGS...,SOY,"[-0.11592403054237366, 0.1274290382862091, -0....",0000000000000000000000000000000001001000000000...,1,0.116024


In [ ]:
# result = data.merge(df[['enzyme', 'substrate', 'scores','Binding']],
#                            on=['enzyme', 'substrate'],
#                            how='left')

# result = data.merge(df[['enzyme', 'substrate', 'scores','Binding']],
#                     on=['enzyme', 'substrate'],
#                     how='outer')

result = data.merge(
    df[["enzyme", "substrate", "scores", "Binding"]],
    on=["enzyme", "substrate"],
    how="outer",
)

result["wei_score"].fillna(0, inplace=True)
result["scores"].fillna(0, inplace=True)

In [ ]:
result = result.sort_values(by="Binding", ascending=False)

In [ ]:
df_unique = result.drop_duplicates(subset=result.columns.difference(["Binding"]))

In [ ]:
result[~result.index.isin(df_unique.index)]

In [ ]:
result = df_unique

In [ ]:
print(result["enzyme"].nunique())
print(result["substrate"].nunique())

In [ ]:
unique_to_arr1 = np.setdiff1d(df["enzyme"].unique(), result["enzyme"].unique())
unique_to_arr2 = np.setdiff1d(result["enzyme"].unique(), df["enzyme"].unique())

In [ ]:
unique_to_arr2

In [ ]:
filtered_rows = result[(result["if_right"] == True) & (result["Binding"] == 1)]

In [ ]:
filtered_rows

In [ ]:
c1name = "C14"
c2name = "C6"
subnamenow = "KEO"


c1 = (
    result[(result["substrate"] == subnamenow) & (result["cc"] == c1name)]["enzyme"]
    .unique()
    .tolist()
)
c2 = (
    result[(result["substrate"] == subnamenow) & (result["cc"] == c2name)]["enzyme"]
    .unique()
    .tolist()
)
intersection = np.intersect1d(c1, c2)
filtered_result = result[
    (result["substrate"] == subnamenow)
    & (result["cc"] == c2name)
    & (~result["enzyme"].isin(intersection))
]

filtered_result.loc[:, "cc"] = c1name
filtered_result.loc[:, "wei_score"] = 0
filtered_result.loc[:, "if_right"] = False

In [ ]:
len(filtered_result)

In [ ]:
result = pd.concat([result, filtered_result])

In [ ]:
result[result["enzyme"] == "CYP71D9"]

In [ ]:
# result.to_pickle(our_data+f'screening2/bin/{queryname}_bu.pkl')

In [ ]:
# result = pd.read_pickle(our_data+f'screening2/bin/{queryname}_bu.pkl')